# Chạy crawler arXiv trên Colab (notebook dùng chung cho cả nhóm)

**Mỗi người dùng bản riêng:** *File → Save a copy in Drive*, rồi mở bản copy của mình.
Notebook không chứa key, mật khẩu hay thông tin của riêng ai: mọi thứ cá nhân điền ở **Bước 1** hoặc để trong *Secrets*.

**Làm một lần trước khi chạy**
1. *Runtime → Change runtime type* → **CPU** → *Save*.
2. Bấm 🔑 (*Secrets*) ở thanh bên trái, thêm (và bật *Notebook access* cho notebook này):
   - `S2_API_KEY`: key Semantic Scholar **của riêng bạn**, tự tạo tự để vào.
     Không có key vẫn chạy được nhưng phần references chậm hơn (3 s mỗi request, hay bị 429 hơn).
   - `ARXIV_PROXIES` (*tuỳ chọn*, chỉ khi bật `DUNG_PROXY`): proxy Webshare của bạn, cách nhau bằng dấu phẩy,
     dạng `http://user:pass@ip:port` hoặc `ip:port:user:pass`.
   - `KAGGLE_API_TOKEN` (*tuỳ chọn*): chỉ khi Kaggle đòi đăng nhập lúc lấy ngày (`NGAY_KAGGLE`); không có thì bước đó
     bỏ qua và ngày được hỏi arXiv ở bước *sweep* (chậm hơn).
3. **Không** dán key hay mật khẩu vào ô code nào.

**⚠️ Dung lượng Drive và an toàn dữ liệu (đọc kỹ)**
- Crawler ghi dữ liệu trên **ổ của máy Colab** (`/content/g11/…`), nhanh hơn Drive rất nhiều. Sau mỗi bước, phần mới được nén
  thành **một file `.tar.gz`** lưu vào `THU_MUC_LUU` trên Drive; runtime chết thì Bước 6 khôi phục từ các file đó rồi chạy tiếp.
  Không còn file tạm nào đi qua Drive (trước đây file tải về nằm trên Drive rồi bị xoá, vào thùng rác và làm đầy Drive).
- Bản lưu tính vào dung lượng của **tài khoản Google bạn chọn ở Bước 2**, kể cả khi lưu vào thư mục nhóm do người khác chia sẻ:
  cỡ 0,5 GB cho mỗi khoảng 5000 ID. Xem trước ở *drive.google.com/settings/storage* (nhớ dọn thùng rác).
- Drive **đầy** thì bước lưu báo lỗi và crawler dừng (mã 1); dữ liệu vẫn còn trên máy Colab nếu runtime còn sống:
  dọn chỗ, dọn thùng rác, rồi chạy lại Bước 6 (phần chưa lưu được lưu ở bước kế tiếp).
- Trong lúc chạy: **không** xoá, đổi tên hay di chuyển thư mục bản lưu trên Drive.
- Chạy xong làm **Bước 9** (đẩy hết lên Drive) rồi mới tắt runtime.

**Cách chạy:** lần lượt từ trên xuống (Shift + Enter). Mỗi kế hoạch chạy (Bước 1) chỉ được **một** runtime chạy tại một lúc. **Không được chạy RUN ALL**

## Bước 1. Điền thông tin (mỗi người chỉ sửa 3 dòng đầu)
- Dữ liệu chạy trên máy Colab; bản lưu nằm trên Drive ở `THU_MUC_LUU/<THANH_VIEN>_<ID đầu>_<ID cuối>`
  (mặc định `03_benchmark/`, tách khỏi dữ liệu cũ trong `02_trial/`, nên lần chạy này bắt đầu từ đầu và cho benchmark sạch).
- `TU`, `DEN` **để trống = chạy cả khoảng giao: đây là bản chính thức.** Điền một đoạn nhỏ (ví dụ 100 ID) = chạy thử,
  ra một thư mục riêng, xoá được sau khi thử xong.
- **Benchmark cả nhóm:** một người điền `CAC_KHOANG` (tên và khoảng của từng thành viên); notebook chạy **lần lượt** từng khoảng,
  mỗi lúc một crawler (điều khoản API arXiv: giới hạn tốc độ tính cho *mọi máy bạn điều khiển cộng lại*). Để trống thì chạy khoảng của bạn.
- Phần **"chung cho cả nhóm"** đã điền sẵn, không cần sửa:
  - `DOAN`: số bài mỗi bước (mỗi bước cỡ 15–30 phút); runtime chết thì chỉ phải làm lại bước đang chạy.
  - `NGAY_KAGGLE`: ngày nộp arXiv của các bài được trích lấy từ bản metadata của arXiv trên Kaggle, thay cho khoảng 700 lần gọi
    API arXiv (3 s mỗi lần) cho mỗi 5000 bài; Kaggle lỗi thì tự quay về hỏi arXiv.
  - `CHO_TOI_DA`: arXiv từ chối liên tục chừng ấy phút thì bước dừng, 15 phút sau tự thử lại một lần.
- `FILE_MOI`: để `"github"` khi code mới đã push lên `BRANCH`. Chưa push thì `"tai_len"` (chọn các file `.py` mới từ máy)
  hoặc đường dẫn một thư mục Drive chứa các file `.py` mới mà người viết code chia sẻ.
- `DUNG_PROXY` / `DUNG_WARP`: đã bật sẵn. Proxy cần secret `ARXIV_PROXIES` của riêng bạn (không có thì tự đi thẳng);
  WARP do notebook tự cài trên Colab. Colab cấm kết nối proxy ngoài: nếu runtime bị ngắt bất thường thì đặt cả hai thành `False`.

In [ ]:
# ===================== MỖI NGƯỜI TỰ ĐIỀN =====================
THANH_VIEN  = ""      # tên của bạn, đứng đầu tên thư mục dữ liệu, ví dụ "AQuan", "GBao"
KHOANG_GIAO = ""      # khoảng được giao, chép 4 cột trong sheet, dạng "YYYY-MM <số đầu> YYYY-MM <số cuối>"
TU, DEN     = "", ""  # để trống = cả khoảng giao (bản chính thức); chạy thử một đoạn thì điền "YYMM.NNNNN"

# ===== BENCHMARK CẢ NHÓM (chỉ người chạy benchmark điền; để trống = chỉ khoảng ở trên) =====
CAC_KHOANG = [
    # ("AQuan", "2026-05 12338 2026-05 17337"),
    # ("TenBan", "YYYY-MM <số đầu> YYYY-MM <số cuối>"),
]

# ============ CHUNG CHO CẢ NHÓM (đã điền sẵn, không cần sửa) ============
NHOM_DIR   = "/content/drive/MyDrive/group11-data"
REPO_URL   = "https://github.com/mituleefam/group11-arxiv-project.git"
BRANCH     = "quan-download-robust"
FILE_MOI   = "github"   # code mới chưa push lên BRANCH thì: "tai_len" | thư mục Drive chứa các file .py mới
DUNG_PROXY = True       # cần secret ARXIV_PROXIES (Webshare của riêng bạn); không có thì tự đi thẳng mạng Colab
DUNG_WARP  = True       # notebook tự cài WARP trên Colab ở Bước 4, không cần cài gì trên máy mình
THU_MUC_LUU = "03_benchmark"  # thư mục giữ bản lưu .tar.gz trên Drive (trong NHOM_DIR)
DOAN        = 1000            # số bài mỗi bước; runtime chết thì chỉ làm lại bước đang chạy
NGAY_KAGGLE = True            # ngày nộp của bài được trích: lấy từ Kaggle thay vì API arXiv
CHO_TOI_DA  = 30              # phút arXiv từ chối liên tục thì dừng bước (tự thử lại 1 lần sau 15 phút)
# =========================================================================

## Bước 2. Kết nối Drive và kiểm tra thông tin
Ô 1 hiện cửa sổ: chọn đúng tài khoản có thư mục dữ liệu của nhóm (`NHOM_DIR`) → *Allow*.
**Bản lưu tính vào dung lượng của tài khoản bạn chọn ở đây** (xem cảnh báo ở đầu notebook).
Nếu thư mục đó do người khác chia sẻ: trên Drive bấm chuột phải → *Thêm lối tắt vào Drive* (vào *My Drive*).
Ô 2 kiểm tra Bước 1, ghi kế hoạch chạy (`/content/g11/plan.json`) và cho biết mỗi khoảng đã có bao nhiêu bản lưu trên Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import glob, json, os, re

def dung(msg):
    raise SystemExit("⚠️ " + msg)

def doc_id(s):
    """'YYMM.NNNNN' -> (YYMM, NNNNN); None nếu sai dạng."""
    k = re.fullmatch(r"\s*(\d{4})\.(\d{4,5})\s*", s or "")
    return (int(k[1]), int(k[2])) if k else None

def ma(t):
    yymm, n = t
    return f"{yymm:04d}.{n:05d}" if yymm >= 1501 else f"{yymm:04d}.{n:04d}"

def doc_khoang(ten, khoang, tu_s="", den_s=""):
    """Một khoảng: tên, 4 cột trong sheet và (tuỳ chọn) đoạn chạy thử TU..DEN."""
    ten = ten.strip()
    if not re.fullmatch(r"[A-Za-z0-9_-]+", ten):
        dung(f'Tên "{ten}": điền THANH_VIEN (hoặc tên trong CAC_KHOANG), ví dụ "AQuan" (chỉ chữ không dấu, số, - và _)')
    k = re.fullmatch(r"\s*(\d{4})-(\d{2})\s+(\d+)\s+(\d{4})-(\d{2})\s+(\d+)\s*", khoang)
    if not k:
        dung(f'{ten}: khoảng phải có dạng "YYYY-MM <số đầu> YYYY-MM <số cuối>" (4 cột trong sheet)')
    dau, cuoi = (int(k[1][2:] + k[2]), int(k[3])), (int(k[4][2:] + k[5]), int(k[6]))
    if dau > cuoi:
        dung(f"{ten}: ID đầu phải đứng trước ID cuối")
    tu = doc_id(tu_s) if tu_s.strip() else dau
    den = doc_id(den_s) if den_s.strip() else cuoi
    if tu is None or den is None:
        dung('TU và DEN phải có dạng "YYMM.NNNNN" (hoặc để trống cả hai)')
    if not dau <= tu <= den <= cuoi:
        dung(f"Cần {ma(dau)} <= TU <= DEN <= {ma(cuoi)} (nằm trong khoảng giao)")
    goc = f"{ten}_{ma(tu).replace('.', '-')}_{ma(den).replace('.', '-')}"
    return {"name": ten, "start": ma(tu), "end": ma(den), "ca_khoang": (tu, den) == (dau, cuoi),
            "tu": tu, "den": den, "so_id": den[1] - tu[1] + 1 if tu[0] == den[0] else None,
            "data_dir": f"/content/g11/{goc}", "backup_dir": f"{NHOM_DIR}/{THU_MUC_LUU}/{goc}"}

for ten, gia_tri in (("NHOM_DIR", NHOM_DIR), ("REPO_URL", REPO_URL), ("BRANCH", BRANCH)):
    if "<" in gia_tri or not gia_tri.strip():
        dung(f"Điền {ten} ở Bước 1 (thay phần <...>): hỏi người gửi notebook nếu không biết")
if not os.path.isdir(NHOM_DIR):
    dung(f"Không thấy {NHOM_DIR}. Thư mục do người khác chia sẻ: chuột phải -> 'Thêm lối tắt vào Drive' rồi chạy lại.")

RUNS = ([doc_khoang(ten, khoang) for ten, khoang in CAC_KHOANG] if CAC_KHOANG
        else [doc_khoang(THANH_VIEN, KHOANG_GIAO, TU, DEN)])
if len({r["name"] for r in RUNS}) != len(RUNS):
    dung("CAC_KHOANG: mỗi tên chỉ được một lần")
for r in RUNS:
    os.makedirs(r["backup_dir"], exist_ok=True)
    # Danh sách ID (chỉ lập được khi đầu và cuối cùng tháng) để validate.py kiểm tra không sót ID nào
    r["ds_id"] = f"{r['backup_dir']}/ds_id.txt" if r["so_id"] else None
    if r["ds_id"]:
        with open(r["ds_id"], "w", encoding="utf-8") as f:
            f.write("\n".join(ma((r["tu"][0], n)) for n in range(r["tu"][1], r["den"][1] + 1)) + "\n")

os.makedirs("/content/g11", exist_ok=True)
PLAN = "/content/g11/plan.json"
with open(PLAN, "w", encoding="utf-8") as f:
    json.dump({"chunk_size": DOAN, "kaggle_dates": NGAY_KAGGLE, "max_wait_minutes": CHO_TOI_DA,
               "runs": [{k: r[k] for k in ("name", "start", "end", "data_dir", "backup_dir")} for r in RUNS]},
              f, ensure_ascii=False, indent=2)

if CAC_KHOANG:
    print(f"Benchmark cả nhóm: {len(RUNS)} khoảng, chạy lần lượt")
else:
    print(f"Thành viên: {RUNS[0]['name']} | " + ("CẢ khoảng giao: bản chính thức" if RUNS[0]["ca_khoang"] else "một đoạn: chạy thử"))
for r in RUNS:
    luu = glob.glob(f"{r['backup_dir']}/snapshots/*.tar.gz")
    print(f"- {r['name']}: {r['start']} .. {r['end']}" + (f" ({r['so_id']} ID)" if r["so_id"] else " (qua nhiều tháng)"))
    print(f"    chạy trên máy Colab: {r['data_dir']}")
    print(f"    bản lưu trên Drive:  {r['backup_dir']} ({len(luu)} bản lưu"
          + (": sẽ khôi phục rồi chạy tiếp)" if luu else ")"))
print("Kế hoạch:", PLAN)
DATA = RUNS[0]["data_dir"]
print(f'Để theo dõi trong terminal (Bước 6), dán dòng này trước: D="{DATA}"')

## Bước 3. Lấy code
Ô 1 lấy code nhánh `BRANCH` (và trả mọi file về bản GitHub, nên chạy lại bao nhiêu lần cũng được).
Ô 2 nạp file `.py` mới theo `FILE_MOI`. Với `"tai_len"`: chọn **cùng lúc** các file `.py` mới (giữ Ctrl khi bấm);
đừng chọn `proxies.txt`. Bản `references.py` mới lấy references của nhiều bài trong một request và in tiến độ mỗi 100 bài
(chưa push thì nạp cùng; Bước 5 ô 1 cho biết đang dùng bản nào). Runtime mới thì phải nạp lại.

In [ ]:
REPO = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
os.chdir("/content")
if not os.path.isdir(f"{REPO}/.git"):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} checkout -q -- .
    !git -C {REPO} fetch -q origin {BRANCH}
    !git -C {REPO} checkout -q -B {BRANCH} FETCH_HEAD
os.chdir(REPO)
!git log --oneline -1
!pip install -q -r src/requirements.txt

In [ ]:
import glob, shutil

def chep(ten, du_lieu):
    if not ten.endswith(".py"):
        print("bỏ qua", ten, "(chỉ nhận file .py)")
        return
    dich = f"{REPO}/src/{ten}"
    print(("thay " if os.path.exists(dich) else "thêm ") + ten)
    with open(dich, "wb") as f:
        f.write(du_lieu)

if FILE_MOI == "github":
    print(f"Dùng nguyên code nhánh {BRANCH}")
elif FILE_MOI == "tai_len":
    from google.colab import files
    TAM = "/content/tai_len"
    shutil.rmtree(TAM, ignore_errors=True)
    os.makedirs(TAM)
    os.chdir(TAM)  # files.upload() lưu thêm một bản ở thư mục hiện tại: để nó ở đây, không lẫn vào repo
    try:
        up = files.upload()
    finally:
        os.chdir(REPO)
    if not up:
        dung("Chưa chọn file nào: chạy lại ô này")
    for ten, du_lieu in up.items():
        chep(ten, du_lieu)
elif os.path.isdir(FILE_MOI):
    ds = sorted(glob.glob(f"{FILE_MOI.rstrip('/')}/*.py"))
    if not ds:
        dung(f"Không có file .py nào trong {FILE_MOI}")
    for p in ds:
        with open(p, "rb") as f:
            chep(os.path.basename(p), f.read())
else:
    dung(f'FILE_MOI = "{FILE_MOI}" không hợp lệ: dùng "github", "tai_len" hoặc một thư mục có thật trên Drive')

## Bước 4. Proxy và WARP (tuỳ chọn)
Hai ô này chỉ làm gì khi `DUNG_PROXY` / `DUNG_WARP` bật ở Bước 1, và chỉ có tác dụng với `rate_limiter.py` bản có proxy.
- Proxy chỉ dùng cho lệnh gọi **API arXiv**; tải file không qua proxy (đỡ tốn 1 GB của Webshare);
  Semantic Scholar không bao giờ qua proxy (giới hạn của S2 tính theo key, không theo IP).
- WARP: wgcf tạo tài khoản WARP miễn phí, wireproxy mở SOCKS5 ở `127.0.0.1:40000` (cổng `rate_limiter.py` dò lúc crawler
  khởi động). Ô tự kiểm tra `warp=on`; không kết nối được thì tự tắt, crawler đi thẳng mạng Colab.

Đổi công tắc hay sửa secret thì chạy lại Bước 1, 2 (ô 2), 4 và 5.

In [ ]:
from google.colab import userdata

def chuan_hoa(p):
    p = p.strip()
    if p and "://" not in p and p.count(":") == 3:  # dạng Webshare ip:port:user:pass
        ip, port, user, pw = p.split(":")
        p = f"http://{user}:{pw}@{ip}:{port}"
    return p

os.environ.pop("ARXIV_PROXIES", None)
if os.path.isfile(f"{REPO}/src/proxies.txt"):  # rate_limiter.py tự đọc file này: không để nó lẫn vào
    os.remove(f"{REPO}/src/proxies.txt")
if not DUNG_PROXY:
    print("Proxy: tắt (API arXiv đi thẳng mạng Colab)")
else:
    try:
        raw = userdata.get("ARXIV_PROXIES")
        ds = [chuan_hoa(p) for p in raw.replace("\n", ",").replace(" ", ",").split(",") if p.strip()]
        os.environ["ARXIV_PROXIES"] = ",".join(ds)
        print(f"Proxy: đã nạp {len(ds)} proxy")
    except Exception as exc:
        print(f"Proxy: không nạp được ARXIV_PROXIES ({type(exc).__name__}): API arXiv đi thẳng mạng Colab. "
              "Kiểm tra secret và Notebook access.")

In [ ]:
# WARP: wgcf + wireproxy (WireGuard chạy trong userspace, không cần TUN). Phải chạy TRƯỚC Bước 6.
import socket, subprocess, time

WARP_DIR = "/content/warp"
WARP_PORT = 40000
WGCF_URL = "https://github.com/ViRb3/wgcf/releases/download/v2.3.1/wgcf_2.3.1_linux_amd64"
WIREPROXY_URL = "https://github.com/pufferffish/wireproxy/releases/download/v1.1.3/wireproxy_linux_amd64.tar.gz"
SOCKS = f"socks5h://127.0.0.1:{WARP_PORT}"

def cong_mo():
    try:
        with socket.create_connection(("127.0.0.1", WARP_PORT), timeout=0.5):
            return True
    except OSError:
        return False

def tat_warp():
    subprocess.run(["pkill", "-f", "chay_warp.sh"])     # vòng tự khởi động lại
    subprocess.run(["pkill", "-f", "wireproxy -s -c"])
    for _ in range(20):
        if not cong_mo():
            return
        time.sleep(0.25)

def trace():
    r = subprocess.run(["curl", "-s", "--max-time", "15", "-x", SOCKS,
                        "https://www.cloudflare.com/cdn-cgi/trace"], capture_output=True, text=True)
    return dict(line.split("=", 1) for line in r.stdout.splitlines() if "=" in line)

def chay(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode:
        dong = [l.strip() for l in (r.stderr or r.stdout).splitlines() if l.strip()] or ["?"]
        loi = [l for l in dong if any(k in l.lower() for k in ("error", "fail", "denied", "429", "lookup"))]
        raise RuntimeError(f"{os.path.basename(cmd[0])} {cmd[1] if len(cmd) > 1 else ''}: {(loi or dong)[0][-200:]}")

tat_warp()  # chạy lại ô này thì tắt bản cũ trước
if not DUNG_WARP:
    print("WARP: tắt (tải file đi thẳng mạng Colab)")
else:
    try:
        !pip install -q "requests[socks]"
        os.makedirs(WARP_DIR, exist_ok=True)
        wgcf, wp = f"{WARP_DIR}/wgcf", f"{WARP_DIR}/wireproxy"
        if not os.path.isfile(wgcf):
            chay(["curl", "-sSfL", "-o", wgcf, WGCF_URL])
            os.chmod(wgcf, 0o755)
        if not os.path.isfile(wp):
            chay(["curl", "-sSfL", "-o", f"{WARP_DIR}/wp.tar.gz", WIREPROXY_URL])
            chay(["tar", "-xzf", f"{WARP_DIR}/wp.tar.gz", "-C", WARP_DIR, "wireproxy"])
            os.chmod(wp, 0o755)
        if not os.path.isfile(f"{WARP_DIR}/wgcf-profile.conf"):
            chay([wgcf, "register", "--accept-tos"], cwd=WARP_DIR)
            chay([wgcf, "generate"], cwd=WARP_DIR)
        with open(f"{WARP_DIR}/wireproxy.conf", "w") as f:
            f.write(f"WGConfig = {WARP_DIR}/wgcf-profile.conf\n\n[Socks5]\nBindAddress = 127.0.0.1:{WARP_PORT}\n")
        with open(f"{WARP_DIR}/chay_warp.sh", "w") as f:  # wireproxy có tắt giữa chừng thì 2 s sau tự bật lại
            f.write(f"while true; do {wp} -s -c {WARP_DIR}/wireproxy.conf >> {WARP_DIR}/wireproxy.log 2>&1; sleep 2; done\n")
        subprocess.Popen(["bash", f"{WARP_DIR}/chay_warp.sh"], start_new_session=True,
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        info = {}
        for _ in range(10):  # cổng mở chưa đủ: phải đi được ra Internet qua WARP
            time.sleep(2)
            info = trace() if cong_mo() else {}
            if info.get("warp") in ("on", "plus"):
                break
        if info.get("warp") in ("on", "plus"):
            print(f"WARP: đang chạy ở {SOCKS}, IP ra ngoài {info.get('ip')} ({info.get('colo')})")
        else:
            tat_warp()
            print("WARP: không kết nối được (Colab có thể chặn UDP 2408). Đã tắt; crawler tự bỏ qua WARP.")
    except Exception as exc:
        tat_warp()
        print(f"WARP: không bật được ({exc}). Crawler tự bỏ qua WARP, tải đi thẳng mạng Colab.")

## Bước 5. Kiểm tra và nạp key
Ô 1 kiểm tra code và đường đi mà **không gửi gì tới arXiv**. Ô 2 (chỉ khi bật proxy/WARP) thử từng đường qua
`api.ipify.org` và `speed.cloudflare.com`: proxy nào báo **HỎNG** thì xoá khỏi secret rồi chạy lại từ Bước 4;
WARP chậm hơn mạng Colab nhiều thì tắt `DUNG_WARP`. Ô 3 nạp key Semantic Scholar.

In [ ]:
import subprocess, sys
KIEM_TRA = r"""
import inspect, re, sys
sys.path.insert(0, "src")
import downloader, references, rate_limiter as rl
an = lambda p: re.sub(r"//[^@/]*@", "//***@", p) if p else "mạng Colab"
lim = (downloader.run.__kwdefaults__ or {}).get("limiter")
song_song = "workers" in inspect.signature(downloader.run).parameters
if lim is not None:
    ket_noi, nhip = getattr(lim, "connections", 1), getattr(lim, "mean_interval", lim.min_interval)
    print(f"Tải /src/: bộ giới hạn '{lim.name}', {ket_noi} kết nối, trung bình {nhip:g} s/request"
          + (", tải nhiều bài cùng lúc" if song_song else ", tải từng bài một"))
print(f"API arXiv: {rl.ARXIV.min_interval:g} s/request")
print("References:", "nhiều bài mỗi request (/paper/batch), bản mới" if hasattr(references, "plan_groups")
      else "từng bài một, bản cũ (muốn dùng bản mới: nạp references.py mới qua FILE_MOI)")
if hasattr(rl, "PROXIES"):
    print("Đường cho API arXiv:", [an(p) for p in rl.PROXIES])
    print("Đường cho tải file:", [an(p) for p in rl._UNLIMITED_LIST])
    print("Tạm bỏ proxy hỏng:", "CÓ" if hasattr(rl, "_quarantine") else "KHÔNG (nên dùng rate_limiter.py bản đã sửa)")
else:
    print("CODE_CHUA_CO_PROXY: mọi request đi thẳng mạng Colab")
"""
kq = subprocess.run([sys.executable, "-c", KIEM_TRA], capture_output=True, text=True, encoding="utf-8", cwd=REPO)
print(kq.stdout + kq.stderr[-3000:])
if kq.returncode:
    dung("Code lỗi khi nạp: xem thông báo ở trên (thiếu file mới? chạy lại Bước 3)")
if (DUNG_PROXY or DUNG_WARP) and "CODE_CHUA_CO_PROXY" in kq.stdout:
    print("⚠️ DUNG_PROXY/DUNG_WARP đang bật nhưng code này chưa có xoay proxy: nạp rate_limiter.py mới qua FILE_MOI")

In [ ]:
# Thử từng đường (không gửi gì tới arXiv): IP ra ngoài và tốc độ tải. Chỉ chạy khi bật proxy/WARP.
KIEM_DUONG = r"""
import re, sys, time
sys.path.insert(0, "src")
import requests, rate_limiter as rl

def an(p):
    return re.sub(r"//[^@/]*@", "//***@", p) if p else "mạng Colab"

if not hasattr(rl, "PROXIES"):
    sys.exit("Code này chưa có xoay proxy")
print("IP ra ngoài của từng đường cho API arXiv:")
hong = []
for p in rl.PROXIES:
    try:
        ip = requests.get("https://api.ipify.org", timeout=15, proxies={"http": p, "https": p} if p else None).text
        print(f"   OK    {an(p):40} -> {ip}")
    except Exception as e:
        if p:
            hong.append(an(p))
        print(f"   HỎNG  {an(p):40} -> {type(e).__name__}")
print("\nTốc độ tải 10 MB qua các đường dùng để tải file:")
for p in rl._UNLIMITED_LIST:
    try:
        t = time.perf_counter()
        n = len(requests.get("https://speed.cloudflare.com/__down?bytes=10000000", timeout=60,
                             proxies={"http": p, "https": p} if p else None).content)
        print(f"   {an(p):40} {n / 1e6 / (time.perf_counter() - t):6.1f} MB/s")
    except Exception as e:
        print(f"   {an(p):40} lỗi {type(e).__name__}")
if hong:
    print(f"\n{len(hong)} proxy hỏng: {', '.join(hong)}")
    print("-> xoá khỏi secret ARXIV_PROXIES rồi chạy lại từ Bước 4. Proxy hỏng không làm dừng crawler")
    print("   (nó bị bỏ qua 5 phút mỗi lần lỗi), nhưng mỗi lần lỗi tốn một request phải thử lại.")
"""
if not (DUNG_PROXY or DUNG_WARP):
    print("Bỏ qua: proxy và WARP đều tắt")
elif "CODE_CHUA_CO_PROXY" in kq.stdout:
    print("Bỏ qua: code này chưa có xoay proxy")
else:
    kq2 = subprocess.run([sys.executable, "-c", KIEM_DUONG], capture_output=True, text=True, encoding="utf-8", cwd=REPO)
    print(kq2.stdout + kq2.stderr[-2000:])

In [ ]:
# Nạp key Semantic Scholar từ Secrets (không in key ra)
try:
    os.environ["S2_API_KEY"] = userdata.get("S2_API_KEY")
    print("Đã nạp S2_API_KEY")
except Exception as exc:
    os.environ.pop("S2_API_KEY", None)
    print(f"Không nạp được S2_API_KEY ({type(exc).__name__}): vẫn chạy được, phần references chậm hơn")

## Bước 6. Chạy
Để **tab này mở** và **máy không ngủ**. Trong lúc ô này chạy, **đừng chạy ô khác** (sẽ phải xếp hàng chờ).
Ô này chạy `src/chunked_crawl.py` theo kế hoạch ở Bước 2, lần lượt từng khoảng:
1. khôi phục bản lưu từ Drive (nếu có) → 2. discovery cả khoảng → 3. metadata, tải nguồn, references theo từng đoạn `DOAN` bài
→ 4. ngày nộp từ Kaggle → 5. *sweep* cả khoảng: làm lại những gì các đoạn để lại (file arXiv chưa trả, lô lỗi, ngày còn thiếu).

Xong mỗi bước, phần mới được lưu lên Drive (dòng `[…] saved 0003_chunk-….tar.gz`). Mỗi bước chạy dưới `benchmark.py`,
nên thời gian kể cả chờ và thử lại đều được tính. Cuối cùng in bảng `== Ranges ==` và ô kết thúc với mã:

| Mã | Nghĩa | Làm gì |
|---|---|---|
| 0 | Xong mọi khoảng | Sang Bước 7 |
| 4 | Xong, còn vài mục (file arXiv chưa trả được, lô lỗi) | 1–2 giờ sau chạy lại ô này: chỉ làm lại phần còn thiếu |
| 3 | arXiv từ chối liên tục (đã tự thử lại sau 15 phút) | 1–2 giờ sau chạy lại ô này |
| 130 | Bạn bấm ■ | Chạy lại ô này khi muốn: làm tiếp từ bước dở |
| 1 | Một bước lỗi 2 lần, hoặc không lưu được lên Drive | Đọc thông báo ngay phía trên; Drive đầy thì dọn rồi chạy lại |

Những dòng sau là **bình thường**, cứ để chạy:
- `[download] … failed: HTTP 503; put aside for a later pass …`: arXiv chưa trả được file đó; nó được gác lại, không làm
  dừng các file khác, và được thử lại ở bước *sweep* (và lần chạy sau).
- `[arXiv] HTTP 429/503 … pausing all arXiv requests for N s`: arXiv từ chối mọi file, crawler dừng rồi thử lại
  (mỗi lần lâu gấp đôi, tối đa 5 phút); thấy `… requests in a row succeeded: back to …` là đã hồi phục.
- `[proxy] … bỏ qua 5 phút`: proxy đó lỗi, request đi đường khác. Cùng một proxy hiện mãi thì xoá nó khỏi secret.
- Runtime tự ngắt bất thường khi đang bật proxy/WARP: nhiều khả năng do Colab chặn, tắt 2 công tắc ở Bước 1.

**Theo dõi tiến độ:** `[<tên>] chunk k/n`, trong mỗi đoạn `[download] x/y … about N min left` và
`[references] x/y | … | about N min left`; sau dòng `[references] Semantic Scholar done; finishing arXiv date lookups`
là phần tra ngày (không có khi dùng `NGAY_KAGGLE`, trừ bước *sweep*).

In [ ]:
!python src/chunked_crawl.py "{PLAN}"

## Bước 7. Kiểm tra dữ liệu
Ô dưới khôi phục bản lưu nếu đây là runtime mới (không làm gì nếu đã có), rồi kiểm tra từng khoảng. Không có dòng `FAIL` là đạt.
`PENDING` ở phiên bản `not_found` nghĩa là arXiv không có file nguồn cho phiên bản đó (404), không phải lỗi crawler;
`PENDING` ở phiên bản `failed` thì 1–2 giờ sau chạy lại Bước 6 để lấy nốt.
Các phiên bản 404 được ghi vào `not_found_can_kiem_tra.csv` trong thư mục bản lưu của khoảng đó trên Drive. Mở vài trang
`https://arxiv.org/abs/<id><version>` kiểm tra (thường là bản đã rút, không có *TeX Source*); đúng thì đổi tên file thành
`verified_404.csv` rồi chạy lại ô: các dòng đó chuyển thành `WARN` (404 đã kiểm tra), ghi vào báo cáo như một ngoại lệ có ghi nhận.

**Đủ ID chưa:** ô này lập `ds_id.txt` (mọi ID của khoảng) rồi validate đối chiếu với dữ liệu. Khoảng qua nhiều tháng thì
chỗ hết tháng lấy từ bản metadata arXiv trên Kaggle (cùng bản `NGAY_KAGGLE` dùng), không lấy từ kết quả crawler. Có dòng
`FAIL assigned_id_missing` nghĩa là discovery đã bỏ sót các ID đó: chạy ô dưới đây (điền `D`, `LUU` của khoảng đó và ID
thiếu đầu, cuối), rồi chạy lại ô này:
```python
!python src/benchmark.py --data-dir "{D}" --output-dir "{LUU}/perf" -- python src/main.py --start <ID đầu> --end <ID cuối> --data-dir "{D}"
!python src/snapshot.py save --data-dir "{D}" --backup-dir "{LUU}/snapshots" --label bo-sung
```

In [ ]:
import pandas as pd
!python src/chunked_crawl.py "{PLAN}" --restore-only
for r in RUNS:
    D, LUU = r["data_dir"], r["backup_dir"]
    print(f"\n========== {r['name']} ==========")
    # Danh sách mọi ID của khoảng: qua nhiều tháng thì lấy chỗ hết tháng từ Kaggle (độc lập với crawler)
    DS, TU_, DEN_ = f"{LUU}/ds_id.txt", r["start"], r["end"]
    if os.path.isfile(f"{DS}.tmp"):
        os.remove(f"{DS}.tmp")
    !python src/assigned_ids.py --start {TU_} --end {DEN_} --out "{DS}.tmp" --download
    if os.path.isfile(f"{DS}.tmp"):
        os.replace(f"{DS}.tmp", DS)
        DS_ARG = f'--assigned-ids "{DS}"'
    else:
        DS_ARG = ""
        print("⚠️ Không lập được danh sách ID: validate KHÔNG kiểm tra được có sót ID nào (xem dòng ngay trên)")
    V404 = f"{LUU}/verified_404.csv"
    if os.path.isfile(V404):
        DS_ARG += f' --verified-404 "{V404}"'
        print("Dùng danh sách 404 đã kiểm tra:", V404)
    !python src/validate.py --data-dir "{D}" {DS_ARG} --finished --progress-every 500
    v = pd.read_csv(f"{D}/logs/validation_report.csv", dtype=str)
    v.to_csv(f"{LUU}/validation_report.csv", index=False)  # lên Drive cùng bản lưu
    khac = v[v.status != "PASS"]
    print("Tất cả đều PASS" if khac.empty else f"{len(khac)} dòng không phải PASS:")
    if not khac.empty:
        display(khac[["arxiv_id", "version", "check_name", "status", "error_message"]].head(200))
    nf = khac[(khac.status == "PENDING") & khac.error_message.str.contains("not_found", na=False)][["arxiv_id", "version"]]
    if not nf.empty:
        nf.to_csv(f"{LUU}/not_found_can_kiem_tra.csv", index=False)
        print(f"{len(nf)} phiên bản arXiv trả 404: đã ghi {LUU}/not_found_can_kiem_tra.csv để kiểm tra")
    thieu = v[v.check_name == "assigned_id_missing"].arxiv_id.tolist()
    if thieu:
        print(f"⚠️ {len(thieu)} ID trong khoảng không có trong dữ liệu (từ {thieu[0]} đến {thieu[-1]}): xem hướng dẫn ở trên ô này")

## Bước 8. Tốc độ và số lần arXiv từ chối
Bảng của cả nhóm (`group_benchmark.csv`/`.json`) ghi vào `THU_MUC_LUU/tong_ket/` trên Drive; bên dưới là tóm tắt từng khoảng.
Các khoảng chạy lần lượt trên cùng một runtime nên dòng `group` lấy **ổ đĩa tối đa** là lúc các thư mục trên ổ cộng lại
lớn nhất (thư mục của khoảng đã xong vẫn nằm trên ổ), không phải thư mục lớn nhất (`--one-machine`).

In [ ]:
import glob, json
import pandas as pd
!python src/chunked_crawl.py "{PLAN}" --restore-only
TONG = f"{NHOM_DIR}/{THU_MUC_LUU}/tong_ket"
CAC = " ".join(f'{r["name"]}="{r["data_dir"]}"' for r in RUNS)
MOT_MAY = "--one-machine" if len(RUNS) > 1 else ""  # các khoảng chạy lần lượt trên cùng runtime này
if os.path.isfile("src/group_benchmark.py"):
    !python src/group_benchmark.py {CAC} {MOT_MAY} --out "{TONG}"

def doc(p):
    return pd.read_csv(p) if os.path.isfile(p) and os.path.getsize(p) else pd.DataFrame()

for r in RUNS:
    L = f"{r['data_dir']}/logs"
    print(f"\n========== {r['name']} ==========")
    rl_ = doc(f"{L}/run_log.csv")
    if not rl_.empty:
        print("Thời gian theo loại bước (run_log.csv, giây):", rl_.groupby("stage").seconds.sum().round(0).to_dict())
    dl = doc(f"{L}/download_log.csv")
    if not dl.empty:
        cuoi = dl.drop_duplicates(["arxiv_id", "version"], keep="last")
        print("Phiên bản theo trạng thái:", cuoi.status.value_counts().to_dict())
    th = doc(f"{L}/arxiv_throttle_log.csv")
    if th.empty:
        print("arXiv không từ chối lần nào")
    else:
        dem = th.event.value_counts().to_dict()
        print(f"arXiv: {dem.get('pause', 0)} lần dừng mọi request, {dem.get('isolated', 0)} lỗi riêng từng file, "
              f"{dem.get('defer', 0)} lần gác file để thử sau")
    sl = doc(f"{r['backup_dir']}/snapshot_log.csv")
    if not sl.empty:
        print(f"Lưu/khôi phục Drive: {len(sl)} lần, {sl.seconds.sum():.0f} s (ngoài thời gian crawl)")
    !python src/benchmark_runs.py --logs-dir "{L}"

## Bước 8b. Thời gian xử lý một bài (chạy riêng từng bài)
Đề hỏi thời gian xử lý *một* bài. Lượt chạy chính chỉ cho thời gian **trung bình** (tổng wall time / số bài), vì các bài chạy
xen kẽ nhau. Ô này chọn ngẫu nhiên `SO_MAU` bài (cố định `SEED` để chọn lại được đúng mẫu) trong mọi khoảng, rồi chạy
**từng bài một từ đầu đến cuối** trên thư mục mới (`per_paper_benchmark.py`), đo bằng `benchmark.py`.
- Chạy **sau** khi Bước 6 xong: ô này gửi request tới arXiv và Semantic Scholar, không chạy cùng lúc với crawler.
- Thời gian ô này **không** cộng vào wall time của lượt chạy chính. Mất cỡ 1–2 phút mỗi bài.
- Kết quả lưu thẳng lên Drive (`THU_MUC_LUU/paper_timing/`). Đã có `paper_timings.csv` thì ô bỏ qua; muốn đo lại thì xoá thư mục đó.

In [ ]:
import random, sys
from pathlib import Path
sys.path.insert(0, "src")
from validate import load_manifest

SO_MAU, SEED = 20, 11
DO = f"{NHOM_DIR}/{THU_MUC_LUU}/paper_timing"
!python src/chunked_crawl.py "{PLAN}" --restore-only
if os.path.isfile(f"{DO}/paper_timings.csv"):
    print(f"Đã có {DO}/paper_timings.csv: bỏ qua (xoá thư mục đó nếu muốn đo lại)")
else:
    ids = sorted(i for r in RUNS
                 for i, rec in load_manifest(Path(r["data_dir"]) / "work" / "paper_manifest.jsonl")[0].items()
                 if rec.get("status") == "success")
    MAU = sorted(random.Random(SEED).sample(ids, min(SO_MAU, len(ids))))
    print(f"{len(MAU)} bài trong {len(ids):,} bài tồn tại:", " ".join(MAU))
    MAU_S = " ".join(MAU)
    !python src/per_paper_benchmark.py --output-root "{DO}" --ids {MAU_S}
!cat "{DO}/per_paper_summary.json"

## Bước 8c. Xuất số liệu cho Jupyter Book
Ghi các bảng tổng hợp (không có nội dung LaTeX hay tiêu đề reference) vào `THU_MUC_LUU/tong_ket/book_data_m1/` trên Drive.
Tải cả thư mục đó về, chép đè vào `mst1-book/data/m1/` rồi build sách. Chạy sau Bước 7 (cần `validation_report.csv`)
và sau Bước 8b nếu muốn có thời gian một bài.

In [ ]:
!python src/chunked_crawl.py "{PLAN}" --restore-only
CAC = " ".join(f'{r["name"]}="{r["data_dir"]}"' for r in RUNS)
MOT_MAY = "--one-machine" if len(RUNS) > 1 else ""
DS = " ".join(f'{r["name"]}="{r["backup_dir"]}/ds_id.txt"' for r in RUNS if os.path.isfile(f'{r["backup_dir"]}/ds_id.txt'))
DS_ARG = f"--assigned-ids {DS}" if DS else ""
PT = f"{NHOM_DIR}/{THU_MUC_LUU}/paper_timing/paper_timings.csv"
PT_ARG = f'--paper-timings "{PT}"' if os.path.isfile(PT) else ""
SACH = f"{NHOM_DIR}/{THU_MUC_LUU}/tong_ket/book_data_m1"
!python src/export_report_data.py {CAC} {MOT_MAY} {DS_ARG} {PT_ARG} --out "{SACH}"

## Bước 9. Kết thúc: đẩy hết bản lưu lên Drive
Colab ghi vào Drive qua bộ đệm. Chạy ô này **trước khi tắt runtime** để chắc bản lưu cuối cùng đã lên Drive
(sau đó muốn chạy ô khác thì phải chạy lại Bước 2). Rồi mở Drive web xem `THU_MUC_LUU/<khoảng>/snapshots/` có file mới nhất.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print("Đã đẩy hết lên Drive và gỡ Drive. Có thể tắt runtime.")

### Nếu bị ngắt (Colab ngắt, mất mạng, mã 3, bấm ■)
Không mất phần đã lưu: mỗi bước xong là có bản lưu trên Drive. Mở lại notebook (runtime mới), **giữ nguyên Bước 1**
(cùng `THANH_VIEN`, `KHOANG_GIAO`, `TU`, `DEN`, `CAC_KHOANG`), chạy lần lượt Bước 2 → 6 (Bước 3 phải nạp lại file mới nếu dùng `FILE_MOI`).
- Bước 2 phải in `… N bản lưu: sẽ khôi phục rồi chạy tiếp` với N > 0; in `0 bản lưu` là đang trỏ sang thư mục khác, đừng chạy kẻo làm lại từ đầu.
- Bước 6 in `restored N archives` rồi làm tiếp từ bước dở: chỉ bước đang chạy lúc bị ngắt phải làm lại.
  Bước bị ngắt vẫn để lại dấu thời gian (heartbeat trong `perf/` trên Drive): `group_benchmark.py` liệt kê nó trong phần cảnh báo.
- Chỉ chạy **một** tab/runtime mỗi lúc cho một kế hoạch.